In [53]:
from typing import TypedDict, Literal
from langgraph.graph import START, END, StateGraph

In [54]:
#define the shared state
class State(TypedDict):
    question: str
    documents: str
    answer: str

In [55]:
#creating nodes 1
def retrieve_documents(state:State):
    return {
        "documents":"Our delivery takes 3-5 business days"
    }
    

In [56]:
#creating nodes 2
def generate_answer(state:State):
    return{
        "answer":(
            f"Your Question: {state['question']}\n"
            f"Answer: {state['documents']}"
        )
    }
    

In [57]:
#create graph
graph = StateGraph(State)

In [58]:
#add nodes
graph.add_node("retrieve",retrieve_documents)
graph.add_node("generate",generate_answer)

In [59]:
#connect nodes sequentially
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve","generate")
graph.add_edge("generate",END)

In [60]:
#compile the graph
app = graph.compile()

In [61]:
result = app.invoke({
    "question":"How long does it take for delivery?"
})


In [62]:
print(result["answer"])

Your Question: How long does it take for delivery?
Answer: Our delivery takes 3-5 business days


In [63]:
#conditional workflow
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

In [64]:
#defining shared state
class State(TypedDict):
    question: str
    category:str
    answer:str

In [65]:
#defining the nodes
def classify_question(state:State):
    question = state["question"].lower()

    if "delivery" in question:
        category="delivery"
    elif "return" in question:
        category="return"
    else:
        category="other"
    return {"category": category}

In [66]:
def delivery_node(state:State):
    return {
        "answer":"Delivery takes 3-5 business days"
    }

In [67]:
def other_node(state:State):
    return {
        "answer":"Please provide more details about your question"
    }

In [68]:
def return_node(state:State):
    return{
        "answer":"You can return in 7 days"
    }

In [69]:
#choosing the next nodes
def route_question(
    state:State,
) -> Literal["delivery","return","other"]:
    return state["category"]

In [70]:
#creating the graph
graph = StateGraph(State)

graph.add_node("classify",classify_question)
graph.add_node("delivery",delivery_node)
graph.add_node("return",return_node)
graph.add_node("other",other_node)

In [71]:
#connect the nodes
graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    route_question,
    {
        "delivery": "delivery",
        "return": "return",
        "other": "other",
    },
)

graph.add_edge("delivery", END)
graph.add_edge("return", END)
graph.add_edge("other", END)

# 7. Compile and execute
app = graph.compile()

result = app.invoke({
    "question": "How long does delivery take?"
})

print(result["answer"])

Delivery takes 3-5 business days
